# 🏥 Clinical Video Analysis — Treinamento YOLOv8
Treina 4 modelos especializados por contexto clínico:
- **Cirurgia**: detecção de sangramento
- **Consulta**: sinais de dor/desconforto
- **Fisioterapia**: análise de postura e movimento
- **Triagem**: linguagem corporal indicativa de violência

## 1. Instalar Dependências

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'kagglehub', 'ultralytics', 'pyyaml'], check=True)
print('✅ Dependências instaladas')

✅ Dependências instaladas


## 2. Verificar GPU

In [2]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU disponível: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = 0
else:
    print('⚠️  GPU não encontrada — usando CPU (treinamento mais lento)')
    DEVICE = 'cpu'


⚠️  GPU não encontrada — usando CPU (treinamento mais lento)


## 3. Baixar Datasets via KaggleHub

In [3]:
import kagglehub, os
from concurrent.futures import ThreadPoolExecutor, as_completed

DATASETS = {
    'cirurgia':     'francismon/curated-colon-dataset-for-deep-learning',
    'consulta':     'coder98/emotionpain',
    'fisioterapia': 'sulaimanmuhammed/wlu-rehabilitation-posture',
    'triagem':      'simuletic/cctv-aggressive-poses-and-fight-detection-dataset',
}

def download(item):
    context, dataset_id = item
    try:
        path = kagglehub.dataset_download(dataset_id)
        return context, path, None
    except Exception as e:
        return context, None, str(e)

paths = {}
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(download, item): item[0] for item in DATASETS.items()}
    for future in as_completed(futures):
        context, path, error = future.result()
        if error:
            print(f'❌ {context}: {error}')
        else:
            paths[context] = path
            print(f'✅ {context} → {path}')

print(f'\n{len(paths)}/4 datasets baixados')


c:\Users\Xuanzang\dev\tech-challeng-fase-4\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Baixando cirurgia...


100%|██████████| 1.41G/1.41G [00:48<00:00, 31.1MB/s]

Extracting files...


  ✅ cirurgia → C:\Users\Xuanzang\.cache\kagglehub\datasets\francismon\curated-colon-dataset-for-deep-learning\versions\1
Baixando consulta...


...

Extracting files...


  ✅ fisioterapia → C:\Users\Xuanzang\.cache\kagglehub\datasets\sulaimanmuhammed\wlu-rehabilitation-posture\versions\1
Baixando triagem...


100%|██████████| 20.2M/20.2M [00:02<00:00, 9.73MB/s]

Extracting files...


  ✅ triagem → C:\Users\Xuanzang\.cache\kagglehub\datasets\simuletic\cctv-aggressive-poses-and-fight-detection-dataset\versions\1

Datasets disponíveis: ['cirurgia', 'fisioterapia', 'triagem']


## 4. Inspecionar Estrutura dos Datasets

In [ ]:
import os

for context, path in paths.items():
    print(f'\n📁 {context.upper()} — {path}')
    for root, dirs, files in os.walk(path):
        level = root.replace(path, '').count(os.sep)
        if level > 2:
            continue
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        if level <= 1:
            subindent = '  ' * (level + 1)
            imgs = [f for f in files if f.endswith(('.jpg', '.png', '.jpeg'))]
            txts = [f for f in files if f.endswith('.txt')]
            if imgs:
                print(f'{subindent}🖼️  {len(imgs)} imagens')
            if txts:
                print(f'{subindent}📝 {len(txts)} labels')


## 5. Gerar data.yaml para Cada Contexto

In [ ]:
import yaml, os

CLASSES = {
    'cirurgia':     {0: 'normal', 1: 'bleeding', 2: 'polyp', 3: 'ulcer'},
    'consulta':     {0: 'no_pain', 1: 'pain', 2: 'discomfort'},
    'fisioterapia': {0: 'correct', 1: 'wrong_posture', 2: 'fall_risk'},
    'triagem':      {0: 'normal', 1: 'aggressive', 2: 'protective_gesture'},
}

# Detecta automaticamente se o dataset já tem estrutura train/val
def find_split_path(base_path, split):
    candidates = [
        os.path.join(base_path, split, 'images'),
        os.path.join(base_path, 'images', split),
        os.path.join(base_path, split),
        base_path
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return base_path

os.makedirs('data', exist_ok=True)
yamls = {}

for context, classes in CLASSES.items():
    if context not in paths:
        print(f'⚠️  {context} não baixado, pulando...')
        continue

    os.makedirs(f'data/{context}', exist_ok=True)
    base = paths[context]

    config = {
        'path':  base,
        'train': find_split_path(base, 'train'),
        'val':   find_split_path(base, 'val'),
        'test':  find_split_path(base, 'test'),
        'nc':    len(classes),
        'names': classes
    }

    yaml_path = f'data/{context}/data.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

    yamls[context] = yaml_path
    print(f'✅ {yaml_path} criado — {len(classes)} classes: {list(classes.values())}')


## 6. Treinar os Modelos

In [ ]:
from ultralytics import YOLO
import shutil, os

os.makedirs('models/yolov8_custom', exist_ok=True)

# Muda aqui se quiser treinar só um contexto específico
TREINAR = list(yamls.keys())  # ou ex: ['cirurgia']

resultados = {}

for context in TREINAR:
    print(f'\n{"="*55}')
    print(f'  TREINANDO: {context.upper()}')
    print(f'{"="*55}')

    model = YOLO('yolov8n.pt')  # transfer learning

    results = model.train(
        data=yamls[context],
        epochs=50,
        imgsz=640,
        batch=16,
        name=context,
        project='models/yolov8_custom',
        patience=10,
        augment=True,
        device=DEVICE,
        exist_ok=True,
        verbose=False
    )

    best_src = f'models/yolov8_custom/{context}/weights/best.pt'
    best_dst = f'models/yolov8_custom/{context}.pt'

    if os.path.exists(best_src):
        shutil.copy(best_src, best_dst)
        resultados[context] = best_dst
        print(f'✅ Salvo em {best_dst}')
    else:
        print(f'❌ best.pt não encontrado para {context}')


## 7. Avaliar Métricas dos Modelos

In [ ]:
from ultralytics import YOLO

print(f'{'Contexto':<15} {'mAP50':>8} {'mAP50-95':>10} {'Precisão':>10} {'Recall':>8}')
print('-' * 55)

for context, model_path in resultados.items():
    model = YOLO(model_path)
    metrics = model.val(data=yamls[context], verbose=False)
    print(
        f'{context:<15}'
        f'{metrics.box.map50:>8.3f}'
        f'{metrics.box.map:>10.3f}'
        f'{metrics.box.mp:>10.3f}'
        f'{metrics.box.mr:>8.3f}'
    )


## 8. Testar com Imagem Real

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os

os.makedirs('data/samples', exist_ok=True)

for context, model_path in resultados.items():
    # Busca primeira imagem disponível do dataset
    test_img = next(
        (os.path.join(root, f)
         for root, _, files in os.walk(paths[context])
         for f in files if f.endswith(('.jpg', '.png', '.jpeg'))),
        None
    )

    if not test_img:
        print(f'⚠️  Nenhuma imagem encontrada para {context}')
        continue

    model = YOLO(model_path)
    results = model(test_img, conf=0.25)
    out_path = f'data/samples/resultado_{context}.jpg'
    results[0].save(out_path)

    print(f'\n🔍 {context.upper()}')
    print(f'   Imagem: {os.path.basename(test_img)}')
    print(f'   Detecções: {len(results[0].boxes)}')
    display(Image(filename=out_path, width=400))


## 9. Resumo Final

In [ ]:
import os

print('\n📦 MODELOS TREINADOS:')
print('-' * 45)
for context in ['cirurgia', 'consulta', 'fisioterapia', 'triagem']:
    path = f'models/yolov8_custom/{context}.pt'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  ✅ {context:<15} {path} ({size_mb:.1f} MB)')
    else:
        print(f'  ❌ {context:<15} não treinado')

print('\n🚀 Próximo passo: uvicorn app.main:app --reload')
